### Observações:

- As datas obedecem um dos formatos: "YYYY-MM-DD", "MM-DD-YYYY", "DD/MM/YYYY"
- Teve column shift na popularidade: entradas > 1000 removidas
- placeholders identificados: "N/A", "NA", "NULL", "UNKNOWN" e "NÃO INFORMADO"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, upper, trim, regexp_replace, when, coalesce,
    try_to_date, year, row_number, lit, row_number,
    explode, split, initcap, length, round as _round
)

spark.sql("CREATE SCHEMA IF NOT EXISTS rocket_lab.silver")

In [0]:
df = spark.table("rocket_lab.bronze.tb_movies_info")

# Organiza uma linha por id, ordenadas por ingestion_datetime
w = Window.partitionBy("id").orderBy(col("ingestion_datetime").desc())
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

# Limpeza dos status
status_norm = upper(trim(col("status")))                      # deixa maiúsculo e tira espaços
status_norm = regexp_replace(status_norm, "-", " ")           # troca hífen por espaço
status_norm = regexp_replace(status_norm, "\\s+", " ")        # colapsa espaços duplos (se existirem)

# traduz status
df = df.withColumn("status_filme",
    when(status_norm == "RELEASED",         "Lançado")
    .when(status_norm == "POST PRODUCTION", "Pós-Produção")
    .when(status_norm == "IN PRODUCTION",   "Em Produção")
    .when(status_norm == "PLANNED",         "Planejado")
    .when(status_norm == "RUMORED",         "Rumores")
    .when(status_norm == "CANCELED",        "Cancelado")
    .otherwise("Não Informado")) # inclui null

# traduz o header e verifica se a data de lançamento é válida
df = df.withColumn("data_lancamento", coalesce(
    try_to_date(col("release_date"), "yyyy-MM-dd"),
    try_to_date(col("release_date"), "MM-dd-yyyy"),
    try_to_date(col("release_date"), "dd/MM/yyyy")
))

# requisito: coluna com o ano (pode facilitar o consumo final)
df = df.withColumn("ano_lancamento", year(col("data_lancamento")))

# traduz outros headers
df_silver = df.select(
    col("id").alias("id_filme"),
    col("title").alias("titulo"),
    col("original_title").alias("titulo_original"),
    col("data_lancamento"),
    col("runtime").try_cast("int").alias("duracao_minutos"),
    col("original_language").alias("idioma_original"),
    col("status_filme"),
    col("overview").alias("sinopse"),
    col("tagline").alias("frase_divulgacao"),
    col("ano_lancamento")
)

# salva nova tabela traduzida
df_silver.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_info_filmes")

In [0]:
df = spark.table("rocket_lab.bronze.tb_movies_financials")

# func auxiliar. Recebe o nome da coluna e limpa o valor tirando prefixos e negativos e corrigindo a escala
def limpar_valor(c):
    s = upper(trim(col(c)))
    s = regexp_replace(s, "\\$", "")
    s = regexp_replace(s, "USD", "")
    s = trim(s)
    s = when(s.isin("N/A", "NA", "UNKNOWN", "NÃO INFORMADO", ""),
             lit(None).cast("string")).otherwise(s)

    mult = (when(s.rlike("[0-9]K$"), lit(1000))
            .when(s.rlike("[0-9]M$"), lit(1_000_000))
            .when(s.rlike("[0-9]B$"), lit(1_000_000_000))
            .otherwise(lit(1)))

    s = regexp_replace(s, "[KMB]$", "")
    v = s.try_cast("double") * mult

    return when(v > 0, v).otherwise(lit(None)).cast("decimal(18,2)")

# limpa os campos monetários
df = df.withColumn("orcamento_usd", limpar_valor("budget"))
df = df.withColumn("receita_usd",   limpar_valor("revenue"))

# seleciona o mais recente, desempata pelo que tem menos entradas nulas
df = df.withColumn("_nulos",
    when(col("orcamento_usd").isNull(), 1).otherwise(0) +
    when(col("receita_usd").isNull(), 1).otherwise(0))

w = Window.partitionBy("id").orderBy(
    col("ingestion_datetime").desc(),
    col("_nulos").asc()
)
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn", "_nulos")

# pega a cotação mais recente
cotacao = (
    spark.table("rocket_lab.bronze.tb_cotacao_dolar")
    .orderBy(col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)

# cria as colunas e converte as entradas pra real
df = df.withColumn("orcamento_brl", _round(col("orcamento_usd") * lit(cotacao), 2).cast("decimal(18,2)"))
df = df.withColumn("receita_brl",   _round(col("receita_usd")   * lit(cotacao), 2).cast("decimal(18,2)"))
df = df.withColumn("lucro_usd",
    when(col("orcamento_usd").isNotNull() & col("receita_usd").isNotNull(),
         col("receita_usd") - col("orcamento_usd")))
df = df.withColumn("lucro_brl",
    when(col("orcamento_brl").isNotNull() & col("receita_brl").isNotNull(),
         col("receita_brl") - col("orcamento_brl")))

# margem de lucro (%)
df = df.withColumn("margem_lucro_pct",
    when(col("receita_usd").isNotNull() & (col("receita_usd") != 0) & col("lucro_usd").isNotNull(),
         _round(col("lucro_usd") / col("receita_usd") * 100, 2)))

# tradução final
df_silver = df.select(
    col("id").alias("id_filme"),
    col("orcamento_usd"),
    col("receita_usd"),
    col("orcamento_brl"),
    col("receita_brl"),
    col("lucro_usd"),
    col("lucro_brl"),
    col("margem_lucro_pct")
)

df_silver.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_financeiro_filmes")

In [0]:
df = spark.table("rocket_lab.bronze.tb_movies_metrics")

# popularidade: virgula vira ponto, elimina negativo
df = df.withColumn("popularidade",
    regexp_replace(trim(col("popularity")), ",", ".").try_cast("double"))
df = df.withColumn("popularidade",
    when(col("popularidade") < 0, lit(None)).otherwise(col("popularidade")))
# nota média: virgula vira ponto, força [0-10]
df = df.withColumn("nota_media_tmdb",
    regexp_replace(trim(col("vote_average")), ",", ".").try_cast("double"))
df = df.withColumn("nota_media_tmdb",
    when((col("nota_media_tmdb") < 0) | (col("nota_media_tmdb") > 10), lit(None))
    .otherwise(col("nota_media_tmdb")))
# quantidade de votos: elimina negativo
df = df.withColumn("qtd_votos_tmdb", trim(col("vote_count")).try_cast("int"))
df = df.withColumn("qtd_votos_tmdb",
    when(col("qtd_votos_tmdb") < 0, lit(None)).otherwise(col("qtd_votos_tmdb")))
# nota média: virgula vira ponto, elimina negativo
df = df.withColumn("nota_media_imbd",
    regexp_replace(trim(col("averageRating")), ",", ".").try_cast("double"))
df = df.withColumn("nota_media_imbd",
    when((col("nota_media_imbd") < 0) | (col("nota_media_imbd") > 10), lit(None))
    .otherwise(col("nota_media_imbd")))
# quantidade de votos: elimina negativo
df = df.withColumn("qtd_votos_imbd", trim(col("numVotes")).try_cast("int"))
df = df.withColumn("qtd_votos_imbd",
    when(col("qtd_votos_imbd") < 0, lit(None)).otherwise(col("qtd_votos_imbd")))

# dedup: mais recente, desempate por menos nulos
df = df.withColumn("_nulos",
    when(col("popularidade").isNull(), 1).otherwise(0) +
    when(col("nota_media_tmdb").isNull(), 1).otherwise(0) +
    when(col("qtd_votos_tmdb").isNull(), 1).otherwise(0) +
    when(col("nota_media_imbd").isNull(), 1).otherwise(0) +
    when(col("qtd_votos_imbd").isNull(), 1).otherwise(0))

w = Window.partitionBy("id").orderBy(
    col("ingestion_datetime").desc(),
    col("_nulos").asc()
)
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn", "_nulos")

# tradução e armazenamento
df_silver = df.select(
    col("id").alias("id_filme"),
    col("popularidade"),
    col("nota_media_tmdb"),
    col("qtd_votos_tmdb"),
    col("nota_media_imbd"),
    col("qtd_votos_imbd")
)

df_silver.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_metricas_engajamento")

In [0]:
df = spark.table("rocket_lab.bronze.tb_movies_reviews")

# nota: [0,10]
df = df.withColumn("nota_usuariotmp", trim(col("nota")).try_cast("double"))
df = df.withColumn("nota_usuario",
    when((col("nota_usuariotmp") < 0) | (col("nota_usuariotmp") > 10), lit(None))
    .otherwise(col("nota_usuariotmp")))

# comentário: NULL ou só espaço vira "Sem comentário"
df = df.withColumn("comentario_usuario",
    when(col("comentario").isNull() | (trim(col("comentario")) == ""), lit("Sem comentário"))
    .otherwise(col("comentario")))

df_silver = df.select(
    col("id").alias("id_filme"),
    col("nome").alias("nome_usuario"),
    col("nota_usuario"),
    col("comentario_usuario")
)

df_silver.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_avaliacoes_usuarios")

In [0]:
df = spark.table("rocket_lab.bronze.tb_credits_and_tags")

# normaliza separadores: ';' '|' viram ','
df = df.withColumn("genres_norm", regexp_replace(col("genres"), "[;|]", ","))

# explode: uma linha por gênero por filme
df = df.withColumn("genero_raw", explode(split(col("genres_norm"), ",")))

# remove aspas e espaços
df = df.withColumn("genero_limpo", trim(regexp_replace(col("genero_raw"), '"', "")))

# whitelist dos 19 gêneros TMDB
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western"
]

# mapa preservando caixa canônica
gen_df = spark.createDataFrame(
    [(g.upper(), g) for g in generos_validos],
    ["genero_upper", "nome_genero"]
)

df = df.withColumn("genero_upper", upper(col("genero_limpo")))
df = df.join(gen_df, "genero_upper", "inner")

# final: uma linha por (filme, gênero)
df_silver = df.select(
    col("id").alias("id_filme"),
    col("nome_genero")
).distinct()

df_silver.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_generos")

In [0]:
df = spark.table("rocket_lab.bronze.tb_credits_and_tags")

def explodir(coluna, tipo):
    return df.select(
        col("id").alias("id_filme"),
        explode(split(col(coluna), ",")).alias("nome_entidade")
    ).withColumn("tipo_entidade", lit(tipo))

unificado = (
    explodir("cast", "Ator")
    .union(explodir("directors", "Diretor"))
    .union(explodir("writers", "Roteirista"))
    .union(explodir("production_companies", "Produtora"))
)

# limpeza
unificado = unificado.withColumn("nome_entidade",
    trim(regexp_replace(col("nome_entidade"), '"', "")))

# filtro de lixo: precisa ter letra, comprimento deve ser razoável (até 100), não é placeholder (dentre os identificados)
unificado = unificado.filter(
    (length(col("nome_entidade")) > 0) &
    (length(col("nome_entidade")) <= 100) &
    (col("nome_entidade").rlike("[A-Za-zÀ-ÿ]")) &
    (~upper(col("nome_entidade")).isin("N/A", "NA", "NULL", "UNKNOWN", "NÃO INFORMADO"))
)

# capitalização
unificado = unificado.withColumn("nome_entidade", initcap(col("nome_entidade")))

# dedup
unificado = unificado.distinct()

unificado.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_pessoas_empresas")

In [0]:
df = spark.table("rocket_lab.bronze.tb_cotacao_dolar")

# tipagem
df_silver = df.select(
    col("dataHoraCotacao").try_cast("timestamp").alias("data_hora_cotacao"),
    col("cotacaoCompra").try_cast("decimal(10,4)").alias("cotacao_compra")
)

df_silver.write.mode("overwrite").saveAsTable("rocket_lab.silver.tb_cotacao_dolar")